# AKANE E10-3：自分で書かせた文との比べ・「一種か」にも資料を見せる・意味のずれた文を除く

このノートは **材料を取るだけ** です（比べは結果のファイルを読んで別に行います）。計画：sims/e10/PLAN_E10-3.md。

- N の P：E10-1 の部分と全体（9,750 主張）。⑫ Gemma に自分で文を書かせて見せる／⑪f 意味の合う一般的な文だけ見せる。
- K：「一種か」（23,046 主張）。⑩ 辞書の説明文／⑪g 一般的な文／⑫ 自分で書いた文、対照 ⑩s・⑪s。
- 正解はこのノートに入っていません。手で書いた語の一覧は使いません。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**（**ハイRAM** があれば選ぶ）。
2. 鍵マークの `HF_TOKEN` にアクセスを許可（これまでと同じ）。
3. 「ランタイム」→「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
4. 終わると、ドライブの `AKANE_アーキテクチャ/E10-3_results/` に `E103_*.json.gz` ができます。最後のセルが一覧を出します。
5. 時間の見込み：一般的な文の読み込みと文探し 5分、Gemma に書かせる 20〜30分、Gemma に聞く 40〜50分、NLI 10分。合わせて1.5時間ほど。
6. 途中で切れても、もう一度「すべてのセルを実行」すれば、済んだ所（書かせる所は2,000件ごと）は飛ばします。

In [ ]:
import os, json, time, gzip, subprocess, sys, hashlib, random
import numpy as np
from google.colab import drive, userdata
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/AKANE_アーキテクチャ"
OUT = f"{BASE}/E10-3_results"
R102 = f"{BASE}/E10-2_results"                          # E10-2 で見つけた一般的な文
WIKT = f"{BASE}/E10-1_results/E101_wiktionary.json.gz"   # E10-1 で読んだ辞書
os.makedirs(OUT, exist_ok=True)
NLI_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
GEMMA_ID = "google/gemma-2-2b-it"
GKB_ID = "community-datasets/generics_kb"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lemminflect", "datasets"], check=False)
def save_gz(obj, name, d=None):
    p = f"{d or OUT}/{name}.json.gz"
    with gzip.open(p, "wt", encoding="utf-8") as f: json.dump(obj, f, ensure_ascii=False)
    print("保存:", name, round(os.path.getsize(p) / 1e6, 2), "MB"); return p
def load_gz(name, d=None):
    p = f"{d or OUT}/{name}.json.gz"
    if not os.path.exists(p): return None
    with gzip.open(p, "rt", encoding="utf-8") as f: return json.load(f)

In [ ]:
# 中身（sims/e10/e102/e102_core.py と e103/e103_core.py と同じ。手書きの語の一覧はありません）
import re
"""E10-2の中身（Colabと手元の両方で使う）。計画：sims/e10/PLAN_E10-2.md。手で書いた語の一覧は使わない。
⑩ 辞書の説明文を見せてGemmaに聞く／⑩s 関係のない説明文を見せる（対照）
⑪g 一般的な文の資料（GenericsKB）から、物の語と分類の語の両方を含む文を取り、Gemmaに見せて聞く／⑪s 別の主張の文を見せる（対照）
⑪n 同じ文をNLIの前提にして、主張の文が成り立つか（辞書1段の③と同じ形）
語の形（複数形など）は lemminflect（辞書のデータ）で作る。手で書いた語形の一覧は使わない。
"""
import re

K_DEF, K_FACT, MAX_DEF_CHARS = 3, 5, 220
TOKEN = re.compile(r"[a-z0-9][a-z0-9\-']*")


def lookup(WKD, w):
    """E10-1 と同じ探し方：そのまま → 先頭の大文字/小文字を変えて → 'the ' を外して。"""
    for v in (w, w[:1].upper() + w[1:], w[:1].lower() + w[1:], re.sub(r"^[Tt]he ", "", w)):
        if v in WKD:
            return v
    return None


def defs_text(WKD, w, k=K_DEF):
    """ページに出る順に最初の k 個の意味（印による選び分けはしない）。"""
    key = lookup(WKD, w)
    if not key:
        return "(not in the dictionary)"
    ss = [s[0] if isinstance(s, (list, tuple)) else s["def"] for s in WKD[key][:k]]
    return "; ".join(d[:MAX_DEF_CHARS] for d in ss) if ss else "(not in the dictionary)"


def class_term(dom, c, meta):
    """分類の側の語（辞書を引く語・文を探す語）。P は全体の語、D は分類名、G は地域名。"""
    if dom == "P":
        return c
    if dom == "D":
        return meta["labels"][c]["name"]
    return meta["regions"][c]


def def_prompt(x, xd, w, wd, stmt):
    return (f"Dictionary definitions:\n\"{x}\": {xd}\n\"{w}\": {wd}\n\n"
            f"Using these definitions and your own knowledge, is the following statement true? Answer Yes or No only.\n"
            f"Statement: {stmt}")


def fact_prompt(facts, stmt):
    body = "\n".join(f"- {f}" for f in facts) if facts else "(no facts found)"
    return (f"Facts:\n{body}\n\n"
            f"Using these facts and your own knowledge, is the following statement true? Answer Yes or No only.\n"
            f"Statement: {stmt}")


def gemma_q(stmt):
    return f"Is the following statement true? Answer Yes or No only.\nStatement: {stmt}"   # E4〜E10-1と同じ


def tokens(s):
    return TOKEN.findall(s.lower())


def forms(term, infl):
    """語（または語のまとまり）の形の集まり：そのままの形と、最後の語の名詞の変化形（infl＝lemminflect.getAllInflections）。"""
    ws = tokens(term)
    if not ws:
        return set()
    out = {tuple(ws)}
    for vs in infl(ws[-1], upos="NOUN").values():
        for v in vs:
            out.add(tuple(ws[:-1] + [v.lower()]))
    return out


def find_spans(toks, form):
    n = len(form)
    return any(tuple(toks[i:i + n]) == form for i in range(len(toks) - n + 1))


def term_hits(sents, scores, term_forms, log=print, every=500000):
    """一般的な文を score の高い順に並べ、各語（の形のどれか）を含む文の番号の集まりを返す。
    term_forms: {語: {形(語の並び), ...}}。文は一度だけ読む。返す番号は並べ替えた後の番号（小さいほど score が高い）。"""
    order = sorted(range(len(sents)), key=lambda i: -scores[i])
    by_first = {}
    for t, fs in term_forms.items():
        for f in fs:
            by_first.setdefault(f[0], []).append((t, f))
    hits = {t: set() for t in term_forms}
    kept = {}
    for r, i in enumerate(order):
        toks = tokens(sents[i])
        found = False
        for w in set(toks) & by_first.keys():
            for t, f in by_first[w]:
                if len(f) == 1 or find_spans(toks, f):
                    hits[t].add(r); found = True
        if found:
            kept[r] = sents[i]
        if every and r % every == 0:
            log(f"  文 {r:,}/{len(order):,}")
    return hits, kept


"""E10-3の中身（Colabと手元の両方で使う）。計画：sims/e10/PLAN_E10-3.md。手で書いた語の一覧は使わない。
⑫ Gemmaに自分で文を書かせ、その文を ⑪g と同じ形で見せて聞く（外の資料が要るかの比べ）
⑪f 一般的な文のうち、Gemmaが「問いと同じ意味で使われている」と答えた文だけを見せる（縁起の案）
K 「一種か」の問題（E4-2 の 77,326 主張のうち、常識の判定がある物 501 語の全部の主張）
"""

K_SELF, SELF_NEW_TOKENS = 5, 100
LEAD = re.compile(r"^[\s\-\*•\d\.\)\(]+")   # 行の頭の記号・番号を外す（書式の読み方）


def self_prompt(x, w):
    return (f"Write up to five short, general, factual sentences that mention both \"{x}\" and \"{w}\". "
            f"Write one sentence per line and nothing else.")


def parse_lines(text, k=K_SELF):
    out = []
    for line in text.split("\n"):
        s = LEAD.sub("", line).strip()
        if len(s.split()) >= 3:
            out.append(s)
        if len(out) >= k:
            break
    return out


def sense_prompt(stmt, fact, x, w):
    return (f"Statement: {stmt}\nSentence: {fact}\n\n"
            f"In the sentence, are \"{x}\" and \"{w}\" used with the same meanings as in the statement? Answer Yes or No only.")


def k_stmt(x, cname):
    return f"{x[:1].upper() + x[1:]} is a type of {cname}."   # E4-2 と同じ書き方

In [ ]:
SET_N = json.loads(r'''{"meta": {"wholes": {"architecture": "an architecture", "bacteria": "a bacteria", "beef": "a beef", "bridge": "a bridge", "bud": "a bud", "cat": "a cat", "chapel": "a chapel", "chick": "a chick", "clock": "a clock", "deer": "a deer", "dog": "a dog", "elephant": "an elephant", "embassy": "an embassy", "garage": "a garage", "highway": "a highway", "kidney": "a kidney", "lamb": "a lamb", "male": "a male", "marijuana": "a marijuana", "puppy": "a puppy", "receiver": "a receiver", "reed": "a reed", "rifle": "a rifle", "road": "a road", "salmon": "a salmon", "shark": "a shark", "sheep": "a sheep", "shook": "a shook", "tank": "a tank", "wheat": "a wheat"}, "labels": {"cardio": {"name": "cardiovascular system disease", "art": "a"}, "resp": {"name": "respiratory system disease", "art": "a"}, "gi": {"name": "gastrointestinal system disease", "art": "a"}, "nerv": {"name": "nervous system disease", "art": "a"}, "skin": {"name": "skin disease", "art": "a"}, "musc": {"name": "musculoskeletal system disease", "art": "a"}, "urin": {"name": "urinary system disease", "art": "a"}, "endo": {"name": "endocrine system disease", "art": "an"}, "immune": {"name": "immune system disease", "art": "an"}, "blood": {"name": "blood disease", "art": "a"}, "repro": {"name": "reproductive system disease", "art": "a"}, "breast": {"name": "breast disease", "art": "a"}, "chest": {"name": "thoracic disease", "art": "a"}, "metab": {"name": "metabolic disease", "art": "a"}, "infect": {"name": "infectious disease", "art": "an"}, "tumor": {"name": "cancer or other tumor", "art": "a"}, "mental": {"name": "mental disorder", "art": "a"}, "genetic": {"name": "genetic disease", "art": "a"}, "bact": {"name": "bacterial infection", "art": "a"}, "viral": {"name": "viral infection", "art": "a"}, "fungal": {"name": "fungal infection", "art": "a"}, "parasit": {"name": "parasitic infection", "art": "a"}}, "regions": {"Central America": "Central America", "Western Asia": "Western Asia", "Eastern Asia": "Eastern Asia", "Western Europe": "Western Europe", "South-eastern Asia": "South-Eastern Asia", "Northern Africa": "Northern Africa", "Australia and New Zealand": "Australia and New Zealand", "Melanesia": "Melanesia", "Western Africa": "Western Africa", "Polynesia": "Polynesia", "Southern Africa": "Southern Africa", "Southern Europe": "Southern Europe", "Northern Europe": "Northern Europe", "Caribbean": "Caribbean", "Central Asia": "Central Asia", "Micronesia": "Micronesia", "Southern Asia": "Southern Asia", "Middle Africa": "Middle Africa", "Northern America": "Northern America", "Eastern Africa": "Eastern Africa", "Eastern Europe": "Eastern Europe", "South America": "South America"}}, "order": {"P": ["action", "aerial", "annex", "antenna", "anterior", "antique", "apartment", "aperture", "approach", "arch", "ash", "ash-key", "axle", "balance", "bang", "barb", "barrel", "base", "batting", "beak", "beard", "beetle", "bell", "belly", "bend", "bill", "blood", "blue-belly", "boiler", "bolt", "bottle", "bottle-tree", "boulevard", "brain", "brain-stem", "brake", "breast", "brim", "brow", "bulb", "bullet", "burger", "bypass", "canal", "cannabis", "cannon", "cap", "capacitor", "capital", "cardinal", "carpet", "carving", "case", "casing", "cathode", "ceiling", "chart", "cheat", "cheek", "chest", "chin", "choke", "chromosome", "circle", "circuit", "cleaner", "clip", "clone", "club", "coat", "cock", "comb", "cone", "contact", "cope", "coping", "corner", "cornerstone", "corolla", "corona", "corpus", "cortex", "costa", "course", "court", "courtyard", "cradle", "crest", "crossing", "crown", "curve", "dashboard", "dial", "dice", "differential", "digit", "dock", "door", "doorstep", "doorway", "driver", "duct", "ear", "edible", "electrode", "elevator", "extension", "eyeball", "eyebrow", "eyelid", "facial", "factor", "feather", "feature", "fence", "fin", "finder", "flag", "floor", "flooring", "foil", "food", "forehead", "foundation", "foyer", "frame", "funeral-residence", "fungus", "gene", "god", "goose", "graphic", "grid", "groundwork", "gum", "gun-sight", "hair", "half-moon", "hall", "hammer", "header", "heading", "heat", "heating", "hemisphere", "hen", "hind", "hinge", "hood", "hoof", "hoop", "horse-trail", "interchange", "intersection", "iris", "ivy", "jack", "jacket", "jaw", "jelly", "jersey", "jewel", "jimmy", "john", "key", "keystone", "lag", "landing", "lash", "left", "lens", "level", "lid", "lift", "lighthouse", "lip", "lobby", "lobe", "lock", "lounge", "magazine", "mast", "matrix", "mock-up", "monastery", "moss", "moustache", "movement", "mustache", "muzzle", "nail", "nasal", "nib", "nook", "nose", "nucleus", "oculus", "optic", "orb", "orbit", "pad", "palate", "pale", "pane", "panther", "part", "paw", "peak", "pearl-fish", "pelvis", "petal", "photograph", "pier", "pineapple", "placenta", "plain", "plate", "plume", "poison-berry", "portion", "portrait", "product", "punch", "pupil", "quill", "radiator", "ray", "relay", "resistance", "resistor", "retina", "rib", "rim", "ring", "roadside", "rod", "roe", "roof", "room", "rotary", "roundabout", "row", "ruin", "rump", "saddle", "saliva", "sash", "scum", "section", "segment", "serum", "shaft", "shank", "sheep-tick", "shift", "shoot", "short", "shoulder", "sights", "skeleton", "skull", "slide", "smoke", "snout", "soldier-fish", "spit", "springer", "stair", "staircase", "stairway", "stave", "step", "stigma", "stock", "storey", "story", "stretcher", "stud", "style", "sucker", "sushi", "tail", "tandem", "tap", "target", "teeth", "temple", "thick-knee", "threshold", "time-ball", "toe", "tom-tom", "tongue", "tongue-flower", "tornado", "total", "track", "tread", "trigger", "trunk", "tube", "turnaround", "turnout", "turret", "tyre", "upstairs", "vaccine", "vane", "variation", "vas", "vent-hole", "vertical", "vessel", "volleyball", "wall", "wave", "web", "weed", "weeds", "wheel", "widening", "window", "wing", "wire", "work-clothing"], "D": ["acute diarrhea", "aspiration pneumonia", "swine influenza", "Stevens-Johnson syndrome", "adult T-cell leukemia/lymphoma", "abdominal tuberculosis", "alcohol-related birth defects", "organ system cancer", "Dent disease", "vitamin metabolic disorder", "nicotine dependence", "Ogden syndrome", "sensory system cancer", "heavy chain disease", "delta chain disease", "acquired metabolic disease", "Kennedy's disease", "juvenile absence epilepsy", "stuttering", "peeling skin syndrome", "gastrointestinal allergy", "apple allergy", "plum allergy", "carp allergy", "white shrimp allergy", "snail allergy", "cardiac tuberculosis", "congenital diarrhea", "early-onset Parkinson's disease", "vegetable allergy", "mitochondrial Parkinson's disease", "vascular-type bleeding disorder", "bone remodeling disease", "Coronavirus infectious disease", "kidney cortex disease", "chronic asthma", "acute asthma", "nocturnal asthma", "subjective cognitive decline", "parsley allergy", "traumatic brain injury", "Fraser syndrome", "enhanced S-cone syndrome", "Ballard syndrome", "SHORT syndrome", "Gordon Holmes syndrome", "CHILD syndrome", "CK syndrome", "pelvic inflammatory disease", "small intestine cancer", "breast cyst", "stomach cancer", "intellectual disability", "short bowel syndrome", "separation anxiety disorder", "renal hypertension", "kidney failure", "premature menopause", "malignant hypertension", "tooth disease", "borderline personality disorder", "dependent personality disorder", "obsessive-compulsive disorder", "multiple personality disorder", "impulse control disorder", "lens disease", "trench fever", "neonatal respiratory failure", "T cell deficiency", "acute orbital inflammation", "appendix cancer", "appendix lymphoma", "rabies", "expressive language disorder", "acquired night blindness", "Chandler syndrome", "functional gastric disease", "pelvic muscle wasting", "allergic disease", "cartilage disease", "globe disease", "hepatitis A", "diarrhea", "reading disorder", "Stiff-Person syndrome", "color blindness", "secondary Parkinson disease", "nutmeg liver", "excessive tearing", "red-green color blindness", "cerebral malaria", "Masters-Allen syndrome", "frozen shoulder", "Machado-Joseph disease", "cerebral degeneration", "geographic tongue", "primary pulmonary hypertension", "FG syndrome", "Larsen syndrome", "colon carcinoma", "colon lymphoma", "alcohol use disorder", "acute chest syndrome", "childhood absence epilepsy", "epilepsy", "cannabis dependence", "skull cancer", "optic nerve disease", "placenta cancer", "communication disorder", "hepatitis D", "autoimmune hepatitis", "factor XIII deficiency", "factor XII deficiency", "hepatitis", "cervix disease", "lateral sclerosis", "West Nile fever", "multiple sclerosis", "iris disease", "tooth erosion", "alcoholic psychosis", "Gilbert syndrome", "auditory system disease", "narcissistic personality disorder", "long QT syndrome", "familial Mediterranean fever", "substance-related disorder", "spinal cord disease", "plague", "cerebral arterial disease", "anterior compartment syndrome", "heart valve disease", "bile duct disease", "skin cancer", "Alexander disease", "external ear carcinoma", "smooth muscle tumor", "retinal cancer", "Nelson syndrome", "thoracic cancer", "ear cancer", "inner ear cancer", "simple partial epilepsy", "infertility", "inhibited male orgasm", "external ear cancer", "pulmonary valve disease", "composite lymphoma", "adult lymphoma", "benign vascular tumor", "reactive arthritis", "acne", "heart lymphoma", "chest wall lymphoma", "lung lymphoma", "eye lymphoma", "mitochondrial metabolism disease", "B-cell lymphoma", "childhood leukemia", "placenta disease", "cocaine abuse", "cork-handlers' disease", "gum cancer", "upper lip cancer", "smallpox", "intermediate coronary syndrome", "Wilson disease", "Evans' syndrome", "lower gum cancer", "allergic asthma", "Sheehan syndrome", "blue drum syndrome", "peripheral vertigo", "congenital heart block", "lymph node disease", "cocaine dependence"], "G": ["Bani Yas City", "Al Karama", "The Palm Jumeirah", "Al Mizhar First", "Khulm", "Taloqan", "Eslam Qaleh", "Karukh", "Elbasan", "Berat", "Fier", "Peqin", "Luquembo", "N'zeto", "Quipungo", "Kunda dya Baze", "Lomas de Zamora", "Caleta Olivia", "Lobos", "La Falda", "Brigittenau", "Alsergrund", "Mariahilf", "Sankt Leonhard", "Caloundra", "Randwick", "Dandenong North", "Hawthorn", "Banja Luka", "Bugojno", "Doboj", "Cazin", "Savar", "Barishal", "Uttar Char Fasson", "Srimangal", "Forest", "Izegem", "Watermael-Boitsfort", "Oostkamp", "Koudougou", "Orodara", "Korsimoro", "Karankasso", "Stara Zagora", "Kazanlak", "Targovishte", "Dupnitsa", "Ngozi", "Rumonge", "Kayanza", "Bubanza", "Bassila", "Tchaourou", "Kpozoun", "Toukountouna", "Sengkurong", "Seria", "Tutong", "Liang", "Riberalta", "Bermejo", "San Ignacio de Velasco", "Tupiza", "Angra dos Reis", "Serrinha", "Nova Soure", "Campos Belos", "Tlokweng", "Kanye", "Thamaga", "Janeng", "Orsha", "Slonim", "Novogrudok", "Ivatsevichi", "Downtown Vancouver", "Courtenay", "Willowridge-Martingrove-Richview", "Clarence-Rockland", "Bukavu", "Mwene", "Djugu", "Kabare", "Bocaranga", "Batangafo", "Dekoa", "Kabo", "Owando", "Gamboma", "Mouyondzi", "Pokola", "Sankt Gallen", "Sitten", "Yverdon-les-Bains", "Wettingen", "Katiola", "Tanda", "Diourouzon", "Nzianouan", "La Serena", "Renca", "Hacienda La Calera", "Angol", "Kribi", "Tibati", "Figuil", "Makak", "Puqi", "Qiuji", "Shiqiaozi", "Chenlou", "Santa Marta", "San Gil", "Anaime", "Cota", "San Vicente de Moravia", "Quesada", "Esparza", "San Juan de Dios", "Moa", "Placetas", "Calimete", "La Sierpe", "Pilsen", "Opava", "Michle", "Bohnice", "Marzahn", "Bad Kreuznach", "Hille", "Vechelde", "Randers", "Hedensted", "Glostrup", "Middelfart", "Bonao", "Villa Francisca", "Salcedo", "Hato Mayor del Rey", "Sedrata", "Djidiouia", "Grarem", "Charef", "Ambato", "Machala", "Ibarra", "Rosa Zarate", "Tartu", "Haabersti", "Kristiine", "Viljandi", "Rivas-Vaciamadrid", "Chipiona", "Castilleja de la Cuesta", "Carlet", "Kibre Mengist", "Tulu Bolo", "Deder", "Adebaye", "East Helsinki", "Kuopio", "Herttoniemi", "Raahe", "Troyes", "Rosny-sous-Bois", "Savigny-le-Temple", "Le Pontet", "Oyem", "Bitam", "Oyam", "Makokou", "Bath", "Bicester", "Bideford", "Belle Vale", "Sekondi-Takoradi", "Kade", "Begoro", "Nalerigu", "Nema Kunku", "Lamin", "Kotu", "Tanji", "Camayenne", "Fria", "Mandiana", "Moussadou", "Le Gosier", "Petit-Bourg", "Le Moule", "Basse-Terre", "Volos", "Agios Dimitrios", "Mytilene", "Neapoli", "Zacapa", "Yepocapa", "Alotenango", "Tiquisate", "Bairro Militar", "Reino de Bandim", "Reino de Antula", "Plaque", "Whampoa", "Shui Chuen O", "Wo Che", "Mid Levels", "El Progreso", "La Lima", "Macuelizo", "Marcovia", "Karlovac", "Velika Gorica", "Bjelovar", "Liberpolis", "Croix-des-Bouquets", "Jacmel", "Verrettes", "Lenbe", "Nagykanizsa", "Baja", "Keszthely", "Dunaharaszti", "Letterkenny", "Greystones", "Balbriggan", "Lucan", "Ramat Gan", "Jaffa", "Givatayim", "Or Yehuda", "Pallichal", "Kotagiri", "Palavakkam", "Govardhan", "Basrah", "Najaf", "Al Diwaniyah", "Kufa", "Kermanshah", "Sirjan", "Hashtgerd", "Hoveyzeh", "Circoiscrizione IV", "Bacoli", "Villaricca", "Fossano", "Mandeville", "May Pen", "Ocho Rios", "Half Way Tree", "Zarqa", "Irbid", "Marj Al Hamam", "Judita", "Tochigi", "Maruobara", "Yoshioka", "Makubetsu", "Mombasa", "Eldoret", "Garissa", "Kimilili", "Balykchy", "Razzakov", "Iradan", "Tash-Kumyr", "Kampong Cham", "Pursat", "Lumphat", "Banlung", "Rajin", "Hoeyang", "Chunghwa", "Samho-rodongjagu", "Hwaseong-si", "Masan", "Gwangyang", "Tallang-dong", "Ban Khoan", "Xam Nua", "Pakxan", "Sekong", "Aley", "Jounieh", "Ghazieh", "El Hermel", "Jaffna", "Ambalangoda", "Kandana", "Hambantota", "Buchanan", "Ganta", "Harbel", "Saclepea", "Maputsoe", "Mohale's Hoek", "Mafeteng", "Thaba-Tseka", "Kaunas", "Eiguliai", "Naujininkai", "Telsiai", "Daugavpils", "Ventspils", "Valmiera", "Tukums", "Sirte", "Zuwarah", "Ubari", "Brak", "Skhirate", "Tinghir", "Ait Ourir", "Kelaat Mgouna", "Tiraspol", "Comrat", "Drochia", "Codru", "Ambanja", "Hell-Ville", "Didy", "Amboanjo", "Tetovo", "Kochani", "Lipkovo", "Zelino", "Douentza", "Finnkolo", "Kouri", "Goundam", "Kyimyindine", "Aungban", "Zee Kone", "Yesagyo", "Nalayh", "Khovd", "Tsetserleg", "Dzuunmod", "Mbera", "Tevragh Zeina", "Aleg", "Tembedgha", "Beau Bassin-Rose Hill", "Quatre Bornes", "Goodlands", "Baie du Tombeau", "Salima", "Liwonde", "Ntcheu", "Monkey Bay", "Agua Prieta", "La Barca", "Zacoalco de Torres", "Libres", "Hulu Langat", "Dungun", "Sungkai", "Kampung Tanjung Karang", "Monapo", "Gondola", "Namacurra", "Pebane", "Rundu", "Swakopmund", "Oshakati", "Ondangwa", "Agadez", "Nguigmi", "Mayahi", "Torodi", "Abeokuta", "Agege", "Nguru", "Ipoti", "Chinandega", "Matagalpa", "Mateare", "Juigalpa", "Nieuwegein", "Hoogeveen", "Ommoord", "Zeewolde", "Nesttun", "Halden", "Harstad", "Kongsberg", "Madhyapur Thimi", "Panauti", "Siraha", "Dailekh", "Porirua", "Whangarei", "Paraparaumu", "Karori", "Rustaq", "Sohar", "Sur", "Adam", "La Chorrera", "Chilibre", "Changuinola", "Sabanitas", "Talara", "Abancay", "Picsi", "Quillabamba", "Brookes Point", "Itogon", "Jaen", "Pililla", "Dera Ismail Khan", "Phalia", "Sillanwali", "Harappa", "Konin", "Nysa", "Krapkowice", "Osowa", "Caguas", "Trujillo Alto", "Barceloneta", "Aguadilla", "Hebron", "Old City", "Surif City", "Jericho", "Vila Nova de Gaia", "Arrentela", "Pontinha", "Baguim do Monte", "Fernando de la Mora", "Villa Hayes", "Coronel Oviedo", "Loma Plata", "Turda", "Lugoj", "Gherla", "Gura Humorului", "Novi Pazar", "Kikinda", "Sremska Mitrovica", "Vrbas", "Cyangugu", "Nyabitekeli", "Kayonza", "Umutara", "Madinah", "Sakakah", "King Faisal Military City", "Afif", "Kassala", "El Geneina Fort", "Kadugli", "Al Hasaheisa", "Vasastaden", "Tumba", "Sundbyberg", "Gustavsberg", "Kangkar", "Senja", "Telok Blangah", "Tiong Bahru Estate", "Maribor", "Kranj", "Novo Mesto", "Trbovlje", "Trnava", "Michalovce", "Bardejov", "Malacky", "Kenema", "Port Loko", "Lunsar", "Goderich", "Kolda", "Mboro", "Kayar", "Kbombole", "Berbera", "Balanbale", "Dhabad", "Ceerigaabo", "Pontbuiten", "Rainville", "Lelydorp", "Tammenga", "Malakal", "Wau", "Aweil", "Kafia Kingi", "Santa Tecla", "Delgado", "San Rafael Oriente", "Sonzacate", "Homs", "Al-Tabqa", "Al Qaryatayn", "Tallkalakh", "Kelo", "Doba", "Pala", "Massaguet", "Bassar", "Sotouboua", "Bafilo", "Tabligbo", "Suan Luang", "Bangkok Yai", "Pran Buri", "Bang Bon Nuea", "Boshkengash", "Norak", "Istiqlol", "Vakhsh", "Balkanabat", "Bayramaly", "Yylanly", "Gubadag", "Menzel Bourguiba", "El Kef", "Jendouba", "Borj Cedria", "Arima", "Diego Martin", "Point Fortin", "Tunapuna", "Mbuguni", "Namanyere", "Ngudu", "Kibara", "Kryukiv", "Debaltseve", "Korostyshiv", "Bakhmach", "Butaleja", "Anaka", "Lwakyomba", "Iryango", "The Villages", "Southglenn", "Wheat Ridge", "Carteret", "Fray Bentos", "Treinta y Tres", "Carmelo", "Young", "Kogon Shahri", "Kitob", "Shofirkon Shahri", "Chelak", "Ciudad Guayana", "La Victoria", "Carora", "La Azulita", "Haiphong", "Phu Quoc", "Lai Vung", "Mui Ne", "Al Bayda", "Ataq", "Ma'rib", "Mocha", "Centurion", "Kroonstad", "Harrismith", "Burgersdorp", "Mufulira", "Luwingu", "Manyinga", "Chiengi", "Chegutu", "Chiredzi", "Gokwe", "Gwanda"]}, "corder": {"P": ["architecture", "bacteria", "beef", "bridge", "bud", "cat", "chapel", "chick", "clock", "deer", "dog", "elephant", "embassy", "garage", "highway", "kidney", "lamb", "male", "marijuana", "puppy", "receiver", "reed", "rifle", "road", "salmon", "shark", "sheep", "shook", "tank", "wheat"], "D": ["cardio", "resp", "gi", "nerv", "skin", "musc", "urin", "endo", "immune", "blood", "repro", "breast", "chest", "metab", "infect", "tumor", "mental", "genetic", "bact", "viral", "fungal", "parasit"], "G": ["Australia and New Zealand", "Caribbean", "Central America", "Central Asia", "Eastern Africa", "Eastern Asia", "Eastern Europe", "Melanesia", "Micronesia", "Middle Africa", "Northern Africa", "Northern America", "Northern Europe", "Polynesia", "South America", "South-eastern Asia", "Southern Africa", "Southern Asia", "Southern Europe", "Western Africa", "Western Asia", "Western Europe"]}, "md5": "bb5538d2f0", "n": 26316}''')
SET_K = json.loads(r'''{"ents": ["abolitionist", "absentee", "accelerometer", "accountant", "accused", "adjuster", "aircraft", "althea", "alumnus", "amigo", "amplifier", "anarchist", "ancestor", "anesthesiologist", "aquarium", "archbishop", "architect", "aristocrat", "astrophysicist", "atheist", "attacker", "avenger", "ax", "azalea", "baboon", "babu", "backpack", "ballerina", "banjo", "bantam", "bayonet", "beagle", "beanie", "bedroom", "beekeeper", "belting", "betel", "betrothed", "bicycle", "biennial", "bimbo", "biographer", "bistro", "boathouse", "boatman", "bobcat", "bodybuilder", "bot", "boyfriend", "brad", "bramble", "brandy", "brazier", "breeder", "briefcase", "broad", "brogan", "buggy", "bulldog", "bulldozer", "bumblebee", "burrito", "businesswoman", "buttercup", "buttermilk", "caddie", "calender", "caliper", "camellia", "candlestick", "capacitor", "capo", "carbine", "cardiologist", "caribou", "celibate", "cello", "cesspool", "chalet", "chateau", "cheetah", "chimpanzee", "chipmunk", "chisel", "choreographer", "chronicler", "chub", "churn", "claimant", "clarence", "clary", "classroom", "clinician", "clockwork", "clover", "coachman", "coauthor", "coffin", "collie", "commissar", "composer", "conch", "condor", "confidante", "conger", "contraceptive", "convalescent", "convert", "corgi", "cougar", "councilman", "countess", "coupe", "cowgirl", "cricketer", "crocodile", "crossbow", "curry", "customer", "cutthroat", "cymbal", "dachshund", "darkroom", "dauphin", "debtor", "deer", "deflector", "delinquent", "denture", "dermatologist", "desk", "dimmer", "dinky", "dirk", "dongle", "dowager", "dragoon", "dreadnought", "dredge", "dropper", "dubbing", "dug", "dynamo", "eclectic", "educator", "egalitarian", "empress", "environmentalist", "equatorial", "erica", "espresso", "ethnic", "evergreen", "exhibitor", "exporter", "extrovert", "falafel", "falcon", "fancier", "fascist", "fife", "finch", "flamingo", "flashlight", "foal", "forceps", "fuchsia", "gasket", "gat", "gavel", "geisha", "genitalia", "georgette", "girder", "gland", "glider", "godmother", "goggles", "goldfish", "grabber", "grader", "granddaughter", "grandma", "grandparent", "grandson", "greenhouse", "greyhound", "grocer", "haggis", "handcuff", "handyman", "hangman", "harmonica", "harrow", "headmaster", "heathen", "helicopter", "hillbilly", "hoarder", "homophobe", "hooch", "houseboat", "housekeeper", "housemate", "huck", "hummingbird", "husband", "hyena", "illustrator", "immigrant", "immune", "importer", "incubator", "incurable", "informer", "injector", "insider", "integrator", "interceptor", "invalid", "investor", "isolationist", "janitor", "jasmine", "jester", "juggler", "kicker", "koala", "krill", "landlord", "larva", "liberator", "librarian", "lilac", "limousine", "liza", "llama", "locksmith", "lodger", "loom", "macho", "maggot", "mahatma", "mako", "mamba", "mammal", "manger", "mangrove", "marc", "margarita", "mariposa", "married", "masjid", "mastiff", "matador", "maul", "meatball", "mensch", "mesquite", "messenger", "midshipman", "migrant", "mimic", "minion", "minivan", "minnow", "minster", "miser", "misfit", "missus", "modernist", "moll", "mongoose", "moped", "moray", "motorboat", "motorcycle", "murine", "mustang", "narc", "narrator", "national", "nawab", "necktie", "neurologist", "neutral", "newsreader", "newt", "notary", "oboe", "ombudsman", "overlord", "paella", "pagoda", "paintbrush", "paisley", "palmyra", "panzer", "papillon", "paramedic", "paranoid", "pasha", "patriot", "pedophile", "pelican", "penis", "peony", "peregrine", "perfectionist", "peripheral", "philanthropist", "physicist", "physiologist", "piccolo", "pickpocket", "piglet", "pinto", "playmaker", "playmate", "pleaser", "plenipotentiary", "plow", "plumber", "plush", "poi", "poncho", "pontifical", "portable", "potion", "preteen", "princess", "probable", "progenitor", "programmer", "protectionist", "prowler", "psychiatrist", "psychotherapist", "pug", "pulley", "puppeteer", "quitter", "racist", "radar", "radiologist", "raglan", "rapier", "ratchet", "rayon", "recliner", "redneck", "retailer", "revisionist", "rhinoceros", "ridley", "rifle", "rival", "rodent", "rogue", "rowan", "rustic", "sabra", "salesperson", "sandwich", "sarcophagus", "sarin", "sauerkraut", "scalpel", "scapegoat", "scion", "scraper", "scythe", "secessionist", "sectional", "semaphore", "senator", "senna", "shaheed", "shed", "sheeting", "sherry", "shim", "ship", "shirt", "shiv", "shogun", "sickle", "sieve", "skiff", "sled", "slob", "snoop", "soccer", "socialist", "socialite", "sofa", "spandex", "spastic", "speakeasy", "spitfire", "spokeswoman", "sprinkler", "sprinter", "stagecoach", "stapler", "starfish", "starling", "starship", "statesman", "steamboat", "steeper", "stepdaughter", "stepfather", "stepmother", "stingray", "stockman", "stoner", "storeroom", "stork", "stray", "succulent", "suiting", "sunflower", "supermodel", "supremacist", "surrealist", "survivalist", "swami", "swine", "swot", "syringe", "tabor", "tailor", "takin", "tambourine", "tartan", "tavern", "teapot", "telegraph", "telescope", "therapist", "thermos", "thwart", "tights", "toby", "toddler", "tongs", "tortoise", "totalitarian", "towel", "transponder", "treasurer", "tricycle", "trident", "tugboat", "tun", "turbine", "tweeter", "undercarriage", "underdog", "undergraduate", "undersecretary", "understudy", "undies", "untouchable", "urchin", "valedictorian", "valerian", "valet", "vaporizer", "vermouth", "veronica", "vise", "visitor", "voyeur", "waitress", "walrus", "wanton", "warden", "warlord", "warrior", "washroom", "watchman", "waveguide", "weatherman", "welder", "wheelchair", "whiskey", "wife", "wildflower", "windowsill", "wombat", "woodpecker", "workstation", "worthy", "wraparound", "wrestler", "wristwatch", "yacht", "yucca", "zebra"], "cats": ["living thing", "animal", "mammal", "bird", "fish", "insect", "reptile", "dog", "plant", "tree", "flower", "fruit", "vegetable", "food", "dish", "drink", "man-made object", "tool", "device", "machine", "equipment", "vehicle", "car", "boat", "aircraft", "weapon", "container", "furniture", "clothing", "footwear", "toy", "musical instrument", "building", "room", "person", "worker", "artist", "athlete", "material", "fabric", "metal", "body part", "organ", "disease", "game", "sport"], "n": 23046}''')
def remake(order, corder, meta):
    """ノートの中で主張の文を作り直す（E10・E10-1・E10-2 と同じ書き方）。"""
    cap = lambda s: s[:1].upper() + s[1:]
    art = lambda w: ("an " if w[:1].lower() in "aeiou" else "a ") + w
    out = []
    for x in order["P"]:
        for W in corder["P"]:
            ph = meta["wholes"][W]
            out.append(["P", x, W, f"{cap(x)} is a part of {ph}.", f"{cap(ph.split(' ', 1)[1])} is a part of {art(x)}."])
    for x in order["D"]:
        for l in corder["D"]:
            lb = meta["labels"][l]
            out.append(["D", x, l, f"{cap(x)} is {lb['art']} {lb['name']}.", None])
    for x in order["G"]:
        for g in corder["G"]:
            out.append(["G", x, g, f"{x} is located in {meta['regions'][g]}.", None])
    return out
ITEMS_N = remake(SET_N['order'], SET_N['corder'], SET_N['meta'])
assert hashlib.md5(json.dumps(ITEMS_N).encode()).hexdigest()[:10] == SET_N['md5'], '主張の作り直しが合わない'
PIDX = [i for i, it in enumerate(ITEMS_N) if it[0] == 'P']
NP = [(ITEMS_N[i][1], ITEMS_N[i][2], ITEMS_N[i][3]) for i in PIDX]   # (x, W, 文)
KI = [(x, c, k_stmt(x, c)) for x in SET_K['ents'] for c in SET_K['cats']]   # (x, 分類名, 文)
print('N の P', len(NP), ' K', len(KI)); print(NP[0]); print(KI[0])

## 1. 辞書と一般的な文を読む（N の P は E10-2 の物、K は新しく探す。GPUなし、5分）

In [ ]:
with gzip.open(WIKT, "rt", encoding="utf-8") as f: WKD = json.load(f)["n"]
print("辞書の見出し:", len(WKD))
F102 = load_gz("E102_facts_N", R102); assert F102 is not None, "E10-2_results/E102_facts_N.json.gz が見つかりません"
FN = [[F102["sents"][str(j)] for j in F102["items"][i]] for i in PIDX]   # N の P の各主張の文
FK = load_gz("E103_facts_K")
if FK is None:
    from lemminflect import getAllInflections
    from datasets import load_dataset
    try:
        ds = load_dataset(GKB_ID, "generics_kb", split="train")
    except Exception as e:
        print("全部の版が読めないので best の版にします:", e); ds = load_dataset(GKB_ID, "generics_kb_best", split="train")
    cols = ds.column_names; scol = [c for c in cols if "sentence" in c][0]
    df = ds.to_pandas(); sents = df[scol].astype(str).tolist()   # 列はリストにしてから使う（E10-2 の直し）
    scores = df["score"].astype(float).tolist() if "score" in cols else [0.0] * len(sents); del df
    TERMS = {}
    for x, c, _ in KI:
        for term in (x, c):
            if term not in TERMS: TERMS[term] = forms(term, getAllInflections)
    print("文の数:", len(sents), "探す語:", len(TERMS))
    t = time.time(); HITS, KEPT = term_hits(sents, scores, TERMS); print(f"文を読んだ {time.time() - t:.0f}秒")
    FK = {"items": [[KEPT[j] for j in sorted(HITS[x] & HITS[c])[:K_FACT]] for x, c, _ in KI], "source": GKB_ID}
    save_gz(FK, "E103_facts_K"); del sents, scores, HITS, KEPT
FK = FK["items"]
print("文が見つかった割合  N の P:", round(float(np.mean([len(f) > 0 for f in FN])), 3), " K:", round(float(np.mean([len(f) > 0 for f in FK])), 3))

## 2. Gemma を読み込み、自分で文を書かせる（⑫、GPU、20〜30分。2,000件ごとに保存）

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
YES = tok.encode("Yes", add_special_tokens=False)[0]; NO = tok.encode("No", add_special_tokens=False)[0]
chat = lambda q: tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
def yes_no(qs, B=32, log_every=200):
    out = []; t = time.time()
    for i in range(0, len(qs), B):
        enc = tok([chat(q) for q in qs[i:i + B]], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            r = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
            out.extend((r[:, YES] - r[:, NO]).cpu().numpy().round(3).tolist()); del r
        if (i // B) % log_every == 0: print(f"  {i}/{len(qs)}  {time.time() - t:.0f}秒")
    return out
def write(qs, B=64):
    enc = tok([chat(q) for q in qs], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        g = gm.generate(**enc, max_new_tokens=SELF_NEW_TOKENS, do_sample=False)
    return tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
def write_all(name, qs, B=64, every=2000):
    done = load_gz(name) or {"text": []}; T = done["text"]; t = time.time()
    while len(T) < len(qs):
        j = len(T); chunk = qs[j:j + every]
        for k in range(0, len(chunk), B): T.extend(write(chunk[k:k + B]))
        save_gz({"text": T, "model": GEMMA_ID}, name); print(f"  {len(T)}/{len(qs)}  {time.time() - t:.0f}秒")
    return T
SELF_N = [parse_lines(s) for s in write_all("E103_self_N", [self_prompt(x, W) for x, W, _ in NP])]
SELF_K = [parse_lines(s) for s in write_all("E103_self_K", [self_prompt(x, c) for x, c, _ in KI])]
print("書いた行の数の平均  N の P:", round(float(np.mean([len(s) for s in SELF_N])), 2), " K:", round(float(np.mean([len(s) for s in SELF_K])), 2))
print(NP[0][2], SELF_N[0])

## 3. 意味の確かめ（⑪f）と、Gemma への問い（GPU、40〜50分。方法ごとに保存）

In [ ]:
# 3-1 N の P の一般的な文ごとに「問いと同じ意味で使われているか」
name = "E103_sense_N"
if load_gz(name) is None:
    qs, owner = [], []
    for i, ((x, W, stmt), fs) in enumerate(zip(NP, FN)):
        for f_ in fs: qs.append(sense_prompt(stmt, f_, x, W)); owner.append(i)
    v = yes_no(qs); per = [[] for _ in NP]
    for i, s in zip(owner, v): per[i].append(s)
    save_gz({"logit": per}, name)
SENSE = load_gz(name)["logit"]
FNF = [[f_ for f_, s in zip(fs, ss) if s > 0] for fs, ss in zip(FN, SENSE)]
print("意味が合うと残った文の割合:", round(sum(map(len, FNF)) / max(1, sum(map(len, FN))), 3))
# 3-2 問いを作る（対照の入れ替えは種1、E10-2 と同じ考え）
rng = random.Random(1)
def derange(L):
    sh = L[:]; rng.shuffle(sh); return dict(zip(sh, sh[1:] + sh[:1]))
sx, sc = derange(sorted({x for x, _, _ in KI})), derange(sorted({c for _, c, _ in KI}))
withf = [i for i, f_ in enumerate(FK) if f_]; sf = derange(withf)
Q = {
    "N_self": [fact_prompt(s, stmt) for (_, _, stmt), s in zip(NP, SELF_N)],
    "N_genF": [fact_prompt(f_, stmt) for (_, _, stmt), f_ in zip(NP, FNF)],
    "K_def": [def_prompt(x, defs_text(WKD, x), c, defs_text(WKD, c), stmt) for x, c, stmt in KI],
    "K_defS": [def_prompt(sx[x], defs_text(WKD, sx[x]), sc[c], defs_text(WKD, sc[c]), stmt) for x, c, stmt in KI],
    "K_gen": [fact_prompt(f_, stmt) for (_, _, stmt), f_ in zip(KI, FK)],
    "K_genS": [fact_prompt(FK[sf[i]] if i in sf else [], stmt) for i, (_, _, stmt) in enumerate(KI)],
    "K_self": [fact_prompt(s, stmt) for (_, _, stmt), s in zip(KI, SELF_K)],
}
print({k: len(v) for k, v in Q.items()}); print(Q["K_def"][0]); print(); print(Q["N_self"][0])
for m, qs in Q.items():
    name = f"E103_gemma_{m}"
    if load_gz(name) is not None: print("済み:", name); continue
    print(name, len(qs)); save_gz({"logit": yes_no(qs), "model": GEMMA_ID}, name)
# 確かめ：K の聞き方なしの値を最初の200個だけもう一度（E4-2 の値と合うか）
if load_gz("E103_check_plain_K") is None:
    save_gz({"logit": yes_no([gemma_q(stmt) for _, _, stmt in KI[:200]])}, "E103_check_plain_K")
del gm; torch.cuda.empty_cache()

## 4. NLI（文ごと。N の P は全部の文と意味の合う文、K は全部の文。GPU、10分）

In [ ]:
ntok = AutoTokenizer.from_pretrained(NLI_ID)
nm = AutoModelForSequenceClassification.from_pretrained(NLI_ID).cuda().eval()
lab = {v.lower(): int(k) for k, v in nm.config.id2label.items()}
ORD = [lab["entailment"], lab["contradiction"]]
def nli(P, H, B=128):
    out = np.zeros((len(P), 2), np.float32)
    for i in range(0, len(P), B):
        enc = ntok(P[i:i + B], H[i:i + B], return_tensors="pt", padding=True, truncation=True, max_length=192).to("cuda")
        with torch.no_grad():
            pr = torch.softmax(nm(**enc).logits.float(), -1)[:, ORD]
        out[i:i + len(pr)] = pr.cpu().numpy()
    return out
for name, its, facts in (("E103_nli_N", NP, FN), ("E103_nli_K", KI, FK)):
    if load_gz(name) is not None: print("済み:", name); continue
    P, H, owner = [], [], []
    for i, ((_, _, stmt), fs) in enumerate(zip(its, facts)):
        for f_ in fs: P.append(f_); H.append(stmt); owner.append(i)
    pr = nli(P, H) if P else np.zeros((0, 2)); per = [[] for _ in its]
    for e, i in zip(pr[:, 0].round(4).tolist(), owner): per[i].append(e)
    save_gz({"ent": per, "model": NLI_ID}, name)

## 5. できたファイル

In [ ]:
for f in sorted(os.listdir(OUT)):
    print(f, round(os.path.getsize(f"{OUT}/{f}") / 1e6, 2), "MB")